Definición de tools especializadas y motor determinístico de scoring
para el sistema de evaluación de riesgo en excursiones de montaña.

In [ ]:
#Conexión a Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_path = "/content/drive/MyDrive/hiking_agent_v3"

import os
os.makedirs(project_path, exist_ok=True)
%cd /content/drive/MyDrive/hiking_agent_v3

/content/drive/MyDrive/hiking_agent_v3


In [ ]:
#Estructura en Google Drive
os.makedirs("data", exist_ok=True)
os.makedirs("tools", exist_ok=True)
os.makedirs("core", exist_ok=True)
os.makedirs("agents", exist_ok=True)

In [ ]:
#Crear Tool de Trails y guardarlo en Data como .py
import json

trails = [
    {
        "id": 1,
        "name": "Refugio Laguna Negra",
        "distance_km": 24,
        "elevation_gain_m": 1041,
        "difficulty": "media",
        "time_estimation_hs": 9.5,
        "registration": 1
    },
    {
        "id": 2,
        "name": "Refugio Frey",
        "distance_km": 20,
        "elevation_gain_m": 1024,
        "difficulty": "media",
        "time_estimation_hs": 9,
        "registration": 1
    },
    {
        "id": 3,
        "name": "Refugio Otto Meiling",
        "distance_km": 33,
        "elevation_gain_m": 1330,
        "difficulty": "media-alta",
        "time_estimation_hs": 13,
        "registration": 1
    },
    {
        "id": 4,
        "name": "Mirador Brazo Tristeza",
        "distance_km": 3,
        "elevation_gain_m": 14,
        "difficulty": "fácil",
        "time_estimation_hs": 1.5,
        "registration": 0
    }
]
with open("data/trails.json", "w") as f:
    json.dump(trails, f, indent=2)

In [ ]:
#Crear archivo en tools .py
%%writefile tools/trails_tool.py

import json
from pathlib import Path

DATA_PATH = Path("data/trails.json")

def _load_trails():
    with open(DATA_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


def get_all_trails():
    """Devuelve la lista completa de senderos."""
    return _load_trails()


def get_trail_by_id(trail_id: int):
    """Devuelve un sendero según su ID."""
    trails = _load_trails()

    for trail in trails:
        if trail["id"] == trail_id:
            return trail

    return None


def get_trail_by_name(trail_name: str):
    """Devuelve un sendero según su nombre (case insensitive)."""
    trails = _load_trails()

    for trail in trails:
        if trail["name"].lower() == trail_name.lower():
            return trail

    return None

Overwriting tools/trails_tool.py


In [ ]:
#Crear Tool de Api del clima y guardarlo en Data como .py
%%writefile tools/weather_tool.py
import requests


def get_weather(date: str) -> dict:
    """
    Obtiene clima diario para San Carlos de Bariloche desde Open-Meteo.

    Parámetro:
        date (str): formato 'YYYY-MM-DD'

    Retorna:
        dict con:
            temp_c (float)
            precipitation_mm (float)
            wind_kmh (float)
        o dict con 'error'
        o None si falla la API
    """

    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=-41.1335"
        "&longitude=-71.3103"
        "&daily=temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max"
        "&timezone=America/Argentina/Buenos_Aires"
    )

    try:
        response = requests.get(url, timeout=10)
        data = response.json()
    except Exception:
        return None

    if "daily" not in data:
        return None

    dates = data["daily"]["time"]

    if date not in dates:
        return {"error": "Fecha fuera de rango (máx ~16 días desde hoy)"}

    idx = dates.index(date)

    temp_max = data["daily"]["temperature_2m_max"][idx]
    temp_min = data["daily"]["temperature_2m_min"][idx]

    return {
        "temp_c": round((temp_max + temp_min) / 2, 1),
        "precipitation_mm": data["daily"]["precipitation_sum"][idx],
        "wind_kmh": data["daily"]["windspeed_10m_max"][idx]
    }

Overwriting tools/weather_tool.py


In [ ]:
!pip install astral

In [ ]:
#Tools para calcular hs de luz

%%writefile tools/daylight_tool.py

import datetime as dt
from zoneinfo import ZoneInfo
from astral import LocationInfo
from astral.sun import sun



def get_daylight_hours(date: str) -> dict:

    city = LocationInfo(
        name="Bariloche",
        region="Argentina",
        timezone="America/Argentina/Buenos_Aires",
        latitude=-41.1335,
        longitude=-71.3103
    )

    date_obj = dt.datetime.strptime(date, "%Y-%m-%d").date()

    s = sun(
        city.observer,
        date=date_obj,
        tzinfo=ZoneInfo("America/Argentina/Buenos_Aires")
    )

    sunrise = s["sunrise"]
    sunset = s["sunset"]

    daylight_hours = round(
        (sunset - sunrise).total_seconds() / 3600,
        2
    )

    return {
        "sunrise": sunrise,
        "sunset": sunset,
        "daylight_hours": daylight_hours
    }

Overwriting tools/daylight_tool.py


In [ ]:
#Api a Wikipedia para sumar datos de turismo sobre los trails
%%writefile tools/tourism_tool.py

import requests

HEADERS = {
    "User-Agent": "hiking-agent-v3 (educational project)"
}


def clean_text(text: str) -> str:
    if not text:
        return None

    text = text.strip()
    return text[:1].upper() + text[1:]


def get_tourism_description(trail_name: str) -> dict:
    """
    Busca descripción turística en Wikipedia (español).
    """

    search_query = f"{trail_name} Bariloche"

    search_url = "https://es.wikipedia.org/w/api.php"

    search_params = {
        "action": "query",
        "list": "search",
        "srsearch": search_query,
        "format": "json"
    }

    search_response = requests.get(search_url, params=search_params, headers=HEADERS)

    if search_response.status_code != 200:
        return {
            "title": trail_name,
            "summary": "Error al conectar con Wikipedia.",
            "url": None
        }

    search_data = search_response.json()

    if search_data.get("query", {}).get("search"):
        page_title = search_data["query"]["search"][0]["title"]

        summary_url = f"https://es.wikipedia.org/api/rest_v1/page/summary/{page_title.replace(' ', '_')}"
        summary_response = requests.get(summary_url, headers=HEADERS)

        if summary_response.status_code == 200:
            summary_data = summary_response.json()

            summary = clean_text(summary_data.get("extract"))

            return {
                "title": summary_data.get("title"),
                "summary": summary,
                "url": summary_data.get("content_urls", {}).get("desktop", {}).get("page")
            }

    return {
        "title": trail_name,
        "summary": "No se encontró información en Wikipedia.",
        "url": None
    }

Overwriting tools/tourism_tool.py


In [ ]:
#Score de riesgo

%%writefile core/risk_engine.py

from datetime import timedelta

#----------------------------------#
# EXPERIENCE CLASSIFICATION
def classify_experience(q1, q2, q3):

    if q2 and q3:
        return "avanzado"
    elif q1:
        return "intermedio"
    else:
        return "principiante"


def experience_factor(level):

    if level == "principiante":
        return 3
    elif level == "intermedio":
        return 2
    else:
        return 1

#----------------------------------#
# TRAIL FACTORS

def elevation_factor(elevation_gain_m):
    if elevation_gain_m > 1000:
        return 3
    elif elevation_gain_m > 700:
        return 2
    else:
        return 1


def distance_factor(distance_km):
    if distance_km > 20:
        return 3
    elif distance_km > 12:
        return 2
    else:
        return 1


def difficulty_factor(difficulty):

    mapping = {
        "alta": 3,
        "media-alta": 2.5,
        "media": 2,
        "fácil": 1
    }

    return mapping.get(difficulty, 2)


def registration_factor(registration):
    if registration == 0:
        return 1
    else:
        return 3

#----------------------------------#
# WEATHER

def weather_factor(temp_c, wind_kmh, precipitation_mm):

    if (
        precipitation_mm > 5 or
        wind_kmh > 50 or
        temp_c < 5 or
        temp_c > 30
    ):
        return 3

    elif (
        precipitation_mm > 0 or
        wind_kmh > 30 or
        temp_c < 10 or
        temp_c > 25
    ):
        return 2

    else:
        return 1

#----------------------------------#
# LIGHT


from datetime import datetime, timedelta
from zoneinfo import ZoneInfo  # para zona horaria

def light_factor(start_time_str, duration_hs, sunset_dt):
    """
    start_time_str: string "HH:MM"
    duration_hs: float
    sunset_dt: datetime.datetime (aware)
    """
    # Obtener zona horaria de sunset
    tz = sunset_dt.tzinfo

    # Convertir string de hora de salida a datetime aware
    today = sunset_dt.date()
    start_dt_naive = datetime.strptime(start_time_str, "%H:%M")
    start_dt = start_dt_naive.replace(year=today.year, month=today.month, day=today.day, tzinfo=tz)

    # Calcular hora estimada de llegada
    arrival_time = start_dt + timedelta(hours=duration_hs)

    # Comparar con el sunset
    if arrival_time > sunset_dt:
        return 3
    elif arrival_time > sunset_dt - timedelta(hours=1):
        return 2
    else:
        return 1

#----------------------------------#
# WEIGHTS

WEIGHTS = {
    "elevation": 2.5,
    "distance": 2,
    "weather": 3,
    "light": 3,
    "experience": 3,
    "difficulty": 1.5,
    "registration": 1
}

#----------------------------------#
# RISK ENGINE

def compute_weighted_risk(trail, level, weather_data, light_data, start_time):

    elev = elevation_factor(trail["elevation_gain_m"])
    dist = distance_factor(trail["distance_km"])
    diff = difficulty_factor(trail["difficulty"])
    reg = registration_factor(trail["registration"])
    exp = experience_factor(level)

    weather = weather_factor(
        weather_data["temp_c"],
        weather_data["wind_kmh"],
        weather_data["precipitation_mm"]
    )

    light = light_factor(
        start_time,
        trail["time_estimation_hs"],
        light_data["sunset"]
    )

    total_score = (
        elev * WEIGHTS["elevation"] +
        dist * WEIGHTS["distance"] +
        weather * WEIGHTS["weather"] +
        light * WEIGHTS["light"] +
        exp * WEIGHTS["experience"] +
        diff * WEIGHTS["difficulty"] +
        reg * WEIGHTS["registration"]
    )

    return round(total_score, 2)


def risk_category(score):

    if score < 25:
        return "Bajo"
    elif score < 40:
        return "Moderado"
    elif score < 55:
        return "Alto"
    else:
        return "Crítico"

Overwriting core/risk_engine.py
